<a href="https://colab.research.google.com/github/JulTob/Modelling/blob/main/GLM/Caso_2_07_Isotonico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 07-Cox · Regresión por escalones y rampas

🏵 **Propósito.** Explorar la siniestralidad de la cartera mediante funciones a tramos: **escalones** (niveles constantes) y **rampas** (segmentos lineales), tomando como referencia la práctica de Optimización. La monotonía no se supondrá de antemano.

Conservamos únicamente la preparación y comprobación de la cartera, su inventario de variables y las primeras tasas por exposición y categoría. El ajuste por escalones y rampas se incorporará después, sobre esta base.

**Datos:** cartera del equipo 07-Cox fijada al commit `698dad06c4228ea79319fdba27647a0c61474703`; 3062 pólizas y 40 columnas. Una póliza puede tener varios siniestros y cada póliza tiene un tiempo de exposición diferente.


## 0. Cargar la cartera

Se conserva la fuente fijada que ya estabas usando. `cartera` mantiene sus 40 columnas originales. La conversión de exposición a años se hará en `data`, para que ambas unidades queden identificadas.

In [ ]:
# Preparar las herramientas que usa este cuaderno.
paquetes <- c("tidyverse")
faltantes <- paquetes[!vapply(paquetes, requireNamespace, logical(1), quietly = TRUE)]
if (length(faltantes) > 0) {
  install.packages(faltantes, repos = "https://cloud.r-project.org")
}
suppressPackageStartupMessages(library(tidyverse))

# Fuente de datos del equipo 07: la misma versión del cuaderno de partida.
repositorio <- "https://raw.githubusercontent.com/JulTob/Modelling/698dad06c4228ea79319fdba27647a0c61474703/GLM/"
cartera <- read_csv(paste0(repositorio, "cartera.csv"), show_col_types = FALSE)

stopifnot(
  nrow(cartera) == 3062L, ncol(cartera) == 40L,
  all(c("id_poliza", "n_siniestros", "exposicion", "antiguedad") %in% names(cartera)),
  !anyDuplicated(cartera$id_poliza),
  all(is.finite(cartera$exposicion)), all(cartera$exposicion > 0),
  all(abs(cartera$exposicion * 10 - cartera$antiguedad) < 1e-8)
)

# Apariencia común; cada gráfico conserva sus unidades y su propia pregunta.
options(repr.plot.width = 10, repr.plot.height = 6)
theme_set(theme_minimal(base_size = 12))
glimpse(cartera)
summary(cartera$exposicion)

## 1. Qué representa cada columna

🔰 **El tipo de almacenamiento no determina el significado.** `id_agencia` es un código aunque R lo almacene como número. `bonus_malus_act` es una clase ordenada. `n_siniestros` es un conteo. `exposicion` es tiempo acumulado, expresado en décadas de póliza.

El inventario siguiente recorre los **40 campos** y muestra su familia, clase en R, número de valores distintos y ausencias. Si falta un nombre, la ejecución se detiene: así un error de nombre no hace desaparecer un gráfico silenciosamente.

💡 Los nombres reales son `zona_circulacion`, `tipo_vehiculo` y `color_vehiculo`.

In [ ]:
# Equipo 07-Cox. Primera celda: los 40 campos, por significado.
campos_por_familia <- list(
  identificadores = c("id_poliza", "id_agencia", "id_region"),
  exposicion = c("antiguedad", "exposicion"),
  medidas = c("edad_conductor", "antiguedad_carnet", "potencia_cv", "cilindrada",
              "antiguedad_vehiculo", "valor_vehiculo", "densidad", "km_anuales"),
  categorias = c("edad_tramo", "zona_circulacion", "uso", "tipo_vehiculo",
                 "combustible", "sexo", "color_vehiculo", "estado_civil",
                 "forma_pago", "financiado", "nivel_estudios", "medio_contacto",
                 "bonus_malus_prev", "bonus_malus_act"),
  conteos = c("n_conductores", "n_siniestros", "n_asistencia", "n_fraude", "n_gestiones"),
  tiempos_y_estados = c("t_primer_sin", "evento", "t_baja", "baja",
                       "motivo_fin", "tuvo_siniestro"),
  costes = c("coste_total", "coste_medio")
)
campos <- unlist(campos_por_familia, use.names = FALSE)
stopifnot(length(campos) == 40, !anyDuplicated(campos))
faltan <- setdiff(campos, names(cartera))
sobran <- setdiff(names(cartera), campos)
if (length(faltan) || length(sobran)) {
  stop("Revisar inventario. Faltan: ", paste(faltan, collapse = ", "),
       "; fuera del inventario: ", paste(sobran, collapse = ", "))
}
inventario <- tibble(
  familia = rep(names(campos_por_familia), lengths(campos_por_familia)),
  variable = campos,
  clase_R = vapply(cartera[campos], function(x) paste(class(x), collapse = "/"), character(1)),
  distintos = vapply(cartera[campos], function(x) n_distinct(x, na.rm = TRUE), integer(1)),
  ausentes = vapply(cartera[campos], function(x) sum(is.na(x)), integer(1))
)
print(inventario, n = Inf)
# La clase R es de almacenamiento; la familia expresa el papel de la variable.
# Consulta de una póliza: cambiar solo el identificador.
# cartera %>% filter(id_poliza == 1) %>% glimpse()

## 2. De contar siniestros a medir su ritmo

🔰 Conservamos tus nombres y separamos las dos cantidades:

- $y[i]$: siniestros observados en la póliza $i$.
- $X[i]$: exposición observada, expresada en años.
- $r[i] = y[i]/X[i]$: **tasa observada** de esa póliza.
- $\lambda[i]$: **tasa esperada**, que un modelo estimará a partir del perfil.
- $\mu[i] = E[Y_i\mid\text{perfil},X_i] = X[i]\,\lambda[i]$: número esperado durante su exposición, bajo el supuesto de proporcionalidad.

⚜️ Multiplicar $X[i]$ por $r[i]$ devuelve exactamente $y[i]$: es reconstruir el dato. Para predecir necesitamos estimar $\lambda[i]$ usando un modelo y sus supuestos.

`tasa()` conserva la operación vectorizada que ya usabas. Sus comprobaciones evitan reciclar vectores de longitudes distintas, dividir por cero o tratar una ausencia como si fuera un dato válido.

In [ ]:
# Función de cálculo de tasas observadas, con control de errores.
tasa <- function(eventos, exposicion) {
  stopifnot(
    is.numeric(eventos), is.numeric(exposicion),
    length(eventos) == length(exposicion),
    all(is.finite(eventos)), all(eventos >= 0),
    all(is.finite(exposicion)), all(exposicion > 0)
  )
  eventos / exposicion
}

In [ ]:
# Datos para regresión: conservar todas las variables explicativas.
data <- cartera %>%
  mutate(
    exposicion_anual = 10 * exposicion,
    tasa_siniestros = tasa(n_siniestros, exposicion_anual)
  )

# Comprobar solo las columnas derivadas; no reducir data a ellas.
data %>%
  select(id_poliza, exposicion_anual, n_siniestros, tasa_siniestros) %>%
  head()


### Tasa global: dividir los totales

Un año-póliza es una póliza observada durante un año; dos pólizas observadas medio año suman también un año-póliza.

$$\lambda_{\mathrm{global}} = \frac{\sum_i y[i]}{\sum_i X[i]}
  = \frac{1770}{12882{,}354}
  \approx 0{,}1374\ \text{siniestros por año-póliza}.$$

Son aproximadamente **13,74 siniestros por 100 años-póliza**. Esta cantidad es una tasa, no una probabilidad expresada en porcentaje.

`mean(data$tasa_siniestros)` asigna el mismo peso a cada póliza; el cociente de totales equivale a ponderar cada tasa individual por su exposición. Si ambas medias usan exactamente las mismas filas completas, `mean(y) / mean(X)` también coincide con `sum(y) / sum(X)`, porque los dos tamaños muestrales se cancelan.

In [ ]:
tasa_total <- sum(data$n_siniestros) / sum(data$exposicion_anual)
tasa_total

tibble(
  siniestros = sum(data$n_siniestros),
  anos_poliza = sum(data$exposicion_anual),
  tasa_global = tasa_total,
  media_de_tasas = mean(data$tasa_siniestros),
  media_ponderada = weighted.mean(data$tasa_siniestros, data$exposicion_anual)
)

### Tasas por tipo de vehículo

Comparamos grupos sumando siniestros y exposición **dentro del mismo grupo**. Mostrar también pólizas, siniestros y años-póliza deja visible de dónde sale la tasa.

Las diferencias son descriptivas: pueden intervenir edad, uso, exposición y otras características. Este gráfico no estima un efecto causal ni incluye intervalos de incertidumbre.

In [ ]:
tasas_tipo <- cartera %>% group_by(tipo_vehiculo) %>%
  summarise(polizas = n(), siniestros = sum(n_siniestros),
            anos_poliza = sum(exposicion * 10), .groups = "drop") %>%
  mutate(tasa = siniestros / anos_poliza)
print(tasas_tipo)
ggplot(tasas_tipo, aes(tipo_vehiculo, tasa)) +
  geom_point(size = 3, colour = "steelblue") +
  geom_hline(yintercept = tasa_total, linetype = 2, colour = "tomato") +
  labs(x = "Tipo de vehículo", y = "Siniestros por año-póliza",
       subtitle = "Suma de siniestros / suma de años; línea: tasa global")